In [1]:
import happybase
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T
from datetime import datetime
from tqdm import tqdm
from pyspark.sql import Row
spark = (
    SparkSession.builder
    .appName("testml")
    .master("spark://spark-master:7077")
    .enableHiveSupport()
    .config("hive.metastore.uris", "thrift://hive-metastore:9083")
    .config("spark.cores.max", "1")
    .config("spark.executor.cores", "1")
    .getOrCreate()
)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/12/19 16:10:38 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/12/19 16:10:39 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
25/12/19 16:10:39 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.


In [19]:
spark.sql("SHOW DATABASES").show()

+-----------------+
|        namespace|
+-----------------+
|cryptopredictions|
|          default|
|      test_hive_2|
|           testdb|
+-----------------+



In [20]:
spark.sql("USE cryptopredictions")
spark.sql("SHOW TABLES").show()

+-----------------+--------------------+-----------+
|        namespace|           tableName|isTemporary|
+-----------------+--------------------+-----------+
|cryptopredictions|cryptocurrencysna...|      false|
|cryptopredictions|    usdexchangerates|      false|
|cryptopredictions|       indexsnapshot|      false|
|cryptopredictions|    batch_checkpoint|      false|
+-----------------+--------------------+-----------+



In [31]:
crypto = spark.table("cryptopredictions.cryptocurrencysnapshot")
crypto.show()

+------+-------------------+------------+------------+--------------+---------------+--------------------+-------------+
|Symbol|           Datetime|CurrentPrice|OpeningPrice|LowestDayPrice|HighestDayPrice|PreviousClosingPrice|PartitionDate|
+------+-------------------+------------+------------+--------------+---------------+--------------------+-------------+
|   ETH|2025-11-20 15:04:01|     2988.57|     3079.05|       2873.64|         3106.2|             3079.06|   2025-11-20|
|   ETH|2025-11-20 15:04:01|     2988.57|     3079.05|       2873.64|         3106.2|             3079.06|   2025-11-20|
|   ETH|2025-11-20 15:04:20|     2989.23|      3078.3|       2873.64|         3106.2|              3078.3|   2025-11-20|
|   ETH|2025-11-20 15:04:39|      2986.8|     3078.05|       2873.64|         3106.2|             3078.05|   2025-11-20|
|   SOL|2025-11-20 15:04:58|      140.42|       138.1|        130.53|          144.8|               138.1|   2025-11-20|
|   ETH|2025-11-20 15:05:17|    

In [25]:
index = spark.table("cryptopredictions.indexsnapshot")

In [32]:
index.show()

+---------+-------------------+----------------+-------------+-------------+----------------+----------------+-----------------+------------------+--------------------+--------------------------+-------------------+-----------------------+-----------------------+-------------+
|IndexName|           Datetime|    CurrentPrice|CurrentVolume| OpeningPrice|  LowestDayPrice| HighestDayPrice|LowestYearlyPrice|HighestYearlyPrice|FiftyDayAveragePrice|TwoHundredDaysAveragePrice|TenDayAverageVolume|ThreeMonthAverageVolume|YearOverYearPriceChange|PartitionDate|
+---------+-------------------+----------------+-------------+-------------+----------------+----------------+-----------------+------------------+--------------------+--------------------------+-------------------+-----------------------+-----------------------+-------------+
|      SNP|2025-11-19 15:00:47| 6671.7099609375|    362266710|6625.83984375|6618.47998046875|6675.14990234375|  4835.0400390625|     6920.33984375|   6709.80739257812

In [36]:
import happybase

connection = happybase.Connection(host='hbase')
table = connection.table('crypto_index_aggregates')

rows = []
for key, data in table.scan():
    if b'#1m' in key:
        key_str = key.decode()
        try:
            symbol, ts_str, interval = key_str.split('#')
            timestamp = datetime.strptime(ts_str, "%Y-%m-%d %H:%M:%S")
        except Exception as e:
            # jeśli key nie pasuje do formatu, pomiń
            continue

        row_dict = {
            'symbol': symbol,
            'timestamp': timestamp,
            'interval': interval
        }

        for col, val in data.items():
            col_name = col.decode()
            try:
                row_dict[col_name] = float(val.decode())
            except ValueError:
                row_dict[col_name] = val.decode()

        rows.append(Row(**row_dict))

df = spark.createDataFrame(rows)

df.show(truncate=False)

25/12/19 17:02:29 WARN TaskSetManager: Stage 20 contains a task of very large size (7444 KiB). The maximum recommended task size is 1000 KiB.
[Stage 20:>                                                         (0 + 1) / 1]

+------+-------------------+--------+----------+---------+---------+---------+
|symbol|timestamp          |interval|ohlc:close|ohlc:high|ohlc:low |ohlc:open|
+------+-------------------+--------+----------+---------+---------+---------+
|BTC   |2025-11-06 21:37:00|1m      |100937.93 |100937.93|100937.93|100937.93|
|BTC   |2025-11-06 21:38:00|1m      |100946.74 |100946.74|100922.79|100922.79|
|BTC   |2025-11-06 21:39:00|1m      |100932.99 |100932.99|100918.83|100930.14|
|BTC   |2025-11-06 21:40:00|1m      |100911.99 |100911.99|100898.96|100898.96|
|BTC   |2025-11-06 21:41:00|1m      |100951.99 |100951.99|100924.54|100924.55|
|BTC   |2025-11-06 21:42:00|1m      |101137.41 |101137.41|100976.02|100976.02|
|BTC   |2025-11-06 21:43:00|1m      |101146.47 |101146.47|101120.14|101137.4 |
|BTC   |2025-11-06 21:44:00|1m      |101016.37 |101090.25|101016.37|101090.25|
|BTC   |2025-11-06 21:45:00|1m      |100963.82 |101000.01|100963.82|101000.01|
|BTC   |2025-11-06 21:46:00|1m      |100942.81 |1009

Stock - SNP, DJI, NIM

In [56]:
from pyspark.sql.window import Window
from pyspark.sql.functions import lead, col

symbols_to_keep = ["BTC", "NIM", "SNP", "DJI", "SOL", "ETH"]
df_filtered = df.filter(col("symbol").isin(symbols_to_keep))

window = Window.partitionBy("symbol").orderBy("timestamp")

df_filtered = df_filtered.withColumn("next_close", lead("ohlc:close", 1).over(window))

In [64]:
from pyspark.sql.functions import first, col

symbols = ["BTC", "NIM", "SNP", "DJI", "SOL", "ETH"]

# Pivot ceny
df_close = df_filtered.groupBy("timestamp") \
    .pivot("symbol", symbols) \
    .agg(first("ohlc:close"))

# Pivot next_close z nowymi nazwami
df_next = df_filtered.groupBy("timestamp") \
    .pivot("symbol", symbols) \
    .agg(first("next_close"))

# Zmieniamy nazwy kolumn next_close przed joinem
for sym in symbols:
    if sym in df_next.columns:
        df_next = df_next.withColumnRenamed(sym, f"{sym}_next_close")

# Join po timestamp
df_pivot = df_close.join(df_next, on="timestamp", how="inner")

# Konwersja kolumn na double (oprócz timestamp)
for col_name in df_pivot.columns:
    if col_name != "timestamp":
        df_pivot = df_pivot.withColumn(col_name, col(col_name).cast("double"))
        
feature_cols = ["NIM", "SNP", "DJI", "SOL", "ETH"]
label_col = "BTC_next_close"
df_pivot = df_pivot.dropna(subset=feature_cols + [label_col])
df_pivot.show(5)


25/12/19 18:04:22 WARN TaskSetManager: Stage 129 contains a task of very large size (7444 KiB). The maximum recommended task size is 1000 KiB.
[Stage 132:============================>                            (1 + 1) / 2]

+-------------------+--------+--------------+----------------+--------------+------+-------+--------------+--------------+----------------+--------------+--------------+--------------+
|          timestamp|     BTC|           NIM|             SNP|           DJI|   SOL|    ETH|BTC_next_close|NIM_next_close|  SNP_next_close|DJI_next_close|SOL_next_close|ETH_next_close|
+-------------------+--------+--------------+----------------+--------------+------+-------+--------------+--------------+----------------+--------------+--------------+--------------+
|2025-11-15 11:41:00|95850.01|22900.58984375|6734.10986328125|47147.48046875| 140.7|3161.97|      95811.32|22900.58984375|6734.10986328125|47147.48046875|        140.64|        3161.3|
|2025-11-15 11:43:00|95838.34|22900.58984375|6734.10986328125|47147.48046875|140.59|3161.35|      95807.61|22900.58984375|6734.10986328125|47147.48046875|        140.49|       3159.35|
|2025-11-15 11:45:00|95728.01|22900.58984375|6734.10986328125|47147.4804687

In [68]:
feature_cols = ["NIM", "SNP", "DJI", "SOL", "ETH"]
label_col = "BTC_next_close"

df_ml = df_pivot.dropna(subset=feature_cols + [label_col])

df_ml = VectorAssembler(inputCols=feature_cols, outputCol="features").transform(df_ml)
df_ml = df_ml.select("timestamp", "features", col(label_col).alias("label"))

train_df, test_df = df_ml.randomSplit([0.8, 0.2], seed=42)

lr = LinearRegression(featuresCol="features", labelCol="label")
model = lr.fit(train_df)

predictions = model.transform(test_df)
predictions.select("timestamp", "features", "label", "prediction").show(5)

25/12/19 18:07:16 WARN TaskSetManager: Stage 184 contains a task of very large size (7444 KiB). The maximum recommended task size is 1000 KiB.
25/12/19 18:07:19 WARN TaskSetManager: Stage 191 contains a task of very large size (7444 KiB). The maximum recommended task size is 1000 KiB.
25/12/19 18:07:21 WARN Instrumentation: [4ad4af91] regParam is zero, which might cause numerical instability and overfitting.
25/12/19 18:07:22 WARN TaskSetManager: Stage 201 contains a task of very large size (7444 KiB). The maximum recommended task size is 1000 KiB.
25/12/19 18:07:26 WARN TaskSetManager: Stage 211 contains a task of very large size (7444 KiB). The maximum recommended task size is 1000 KiB.
                                                                                

+-------------------+--------------------+--------+-----------------+
|          timestamp|            features|   label|       prediction|
+-------------------+--------------------+--------+-----------------+
|2025-11-15 11:45:00|[22900.58984375,6...|95722.02|93252.69392034614|
|2025-11-15 11:49:00|[22900.58984375,6...|95686.36|93257.60254774531|
|2025-11-15 11:56:00|[22900.58984375,6...| 95716.7|93215.06559914649|
|2025-11-15 12:08:00|[22900.58984375,6...| 95610.2|93177.48083223987|
|2025-11-15 12:26:00|[22900.58984375,6...| 95876.9|93526.65700401079|
+-------------------+--------------------+--------+-----------------+
only showing top 5 rows



In [69]:
from pyspark.ml.evaluation import RegressionEvaluator

evaluator = RegressionEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="mse"
)

# Liczymy MSE
mse = evaluator.evaluate(predictions)
print(f"MSE: {mse}")

25/12/19 18:08:21 WARN TaskSetManager: Stage 221 contains a task of very large size (7444 KiB). The maximum recommended task size is 1000 KiB.
                                                                                

MSE: 995174.9016139752


In [70]:
evaluator_rmse = RegressionEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="rmse"
)
rmse = evaluator_rmse.evaluate(predictions)
print(f"RMSE: {rmse}")

evaluator_r2 = RegressionEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="r2"
)
r2 = evaluator_r2.evaluate(predictions)
print(f"R2: {r2}")

25/12/19 18:09:02 WARN TaskSetManager: Stage 231 contains a task of very large size (7444 KiB). The maximum recommended task size is 1000 KiB.
                                                                                

RMSE: 997.5845335679454


25/12/19 18:09:05 WARN TaskSetManager: Stage 241 contains a task of very large size (7444 KiB). The maximum recommended task size is 1000 KiB.
                                                                                

R2: 0.8447270251176473


In [71]:
predictions.write.mode("overwrite").saveAsTable("cryptopredictions.btc_predictions")

25/12/19 18:12:16 WARN TaskSetManager: Stage 251 contains a task of very large size (7444 KiB). The maximum recommended task size is 1000 KiB.
25/12/19 18:12:20 WARN SessionState: METASTORE_FILTER_HOOK will be ignored, since hive.security.authorization.manager is set to instance of HiveAuthorizerFactory.


In [77]:
model.write().overwrite().save("hdfs://namenode:8020/models/btc_model")